# 사투리 번역기 — AI Hub 방언 데이터 (부분 다운로드 → 전처리)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysb2152/translate/blob/main/notebooks/aihub_colab.ipynb)

**용량 문제 해결 전략**: AI Hub 방언 데이터는 지역당 수백 GB라 전량은 불필요하고 로컬(34GB)엔 안 들어간다. 여기서는 `aihubshell`로 **필요한 청크(filekey)만 부분 다운로드**해서, 어차피 학습을 돌릴 **Colab에서 직접** 받아 전처리한다. 로컬 디스크는 병목이 안 된다.

**흐름**: aihubshell 설치 → 파일 목록 확인 → 소량 부분 다운로드 → 압축 해제 → 저장소 clone → `data/preprocess.py`로 STT 매니페스트 + (방언→표준) 문장쌍 생성 → Google Drive에 저장.

> 학습(Whisper 파인튜닝)은 데이터가 준비된 다음 셀/노트북에서 이어간다(B-8).

## 0. 준비물

1. **AI Hub 계정** + 받으려는 데이터셋의 **활용 신청 승인**(마이페이지에서 상태 확인).
2. **API 키**: AI Hub 마이페이지 → *API 키 발급*에서 발급받아 아래 `AIHUB_APIKEY`에 넣는다.
3. 런타임: **런타임 → 런타임 유형 변경 → GPU** (전처리 자체는 CPU로 충분하지만, 이어서 학습하려면 GPU).

> 키는 노트북에 하드코딩하지 말고, 실행할 때만 입력하는 걸 권장. 공유 시 반드시 지운다.

In [ ]:
# (선택) Google Drive 마운트 — 전처리 산출물을 세션이 꺼져도 남기려면 사용
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/saturi/processed'
else:
    OUT_ROOT = '/content/processed'
print('산출물 경로:', OUT_ROOT)

In [ ]:
# 설정
import getpass, os

# 데이터셋 키(dataSetSn): 경상도=119, 전라도=120, 충청도=122, 강원도=121, 제주도=123 ...
DATASET_KEY = 119

# API 키(입력 프롬프트로 받음 — 노트북에 저장 안 함)
AIHUB_APIKEY = getpass.getpass('AI Hub API 키 입력: ').strip()

# 작업 폴더(로컬 ephemeral — 빠름). 원본은 여기 받고, 산출물만 Drive로.
RAW_DIR = '/content/aihub_raw'
os.makedirs(RAW_DIR, exist_ok=True)
print('dataset', DATASET_KEY, '| raw', RAW_DIR, '| out', OUT_ROOT)

In [ ]:
# 1) aihubshell 설치
!curl -sL -o /usr/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/bin/aihubshell
!aihubshell -help 2>/dev/null | head -20 || echo '설치 확인: -help 출력이 없으면 URL/네트워크 확인'

In [ ]:
# 2) 파일 목록(filekey)과 용량 확인 → 여기서 받을 청크를 고른다
#    출력의 각 파일 앞 번호가 filekey. 라벨(Training/라벨링) + 그에 대응하는 소량의 원천(음성) 위주로 소량만.
!aihubshell -mode l -datasetkey {DATASET_KEY} -aihubapikey "{AIHUB_APIKEY}"

## 3. 부분 다운로드

위 목록에서 **소량만** 고른다. 팁:
- 라벨(전사 JSON)은 용량이 작으니 필요한 만큼, **원천(음성 WAV)은 한두 청크만** 받아 시작한다.
- 범위 다운로드는 지원 안 하므로 여러 개면 콤마로: `-filekey 44059,44060`.
- 디스크가 빠듯하면 **한 청크 받기 → 전처리 → 원본 삭제 → 다음 청크**를 반복(맨 아래 셀 참고).

아래 `FILEKEYS`를 실제 번호로 바꾼 뒤 실행.

In [ ]:
# 3) 고른 청크만 다운로드 (예시 — 실제 번호로 교체!)
FILEKEYS = 'CHANGE_ME'   # 예: '44059,44060'

assert FILEKEYS != 'CHANGE_ME', '위 목록에서 filekey를 골라 FILEKEYS에 넣으세요.'
!cd {RAW_DIR} && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {FILEKEYS} -aihubapikey "{AIHUB_APIKEY}"
!echo '--- 받은 파일 ---' && find {RAW_DIR} -maxdepth 3 -type f | head -40
!echo '--- 사용량 ---' && du -sh {RAW_DIR}

In [ ]:
# 4) 압축 해제 (AI Hub는 .zip 로 내려옴; 여러 파트로 쪼개진 경우 병합 후 해제)
#    단일 zip 이면 그대로 해제. 멀티파트(.zip + .z01 ...)면 zip -s 0 로 병합.
!apt-get -qq install -y p7zip-full > /dev/null 2>&1
import glob, os, subprocess
for z in glob.glob(f'{RAW_DIR}/**/*.zip', recursive=True):
    print('unzip:', z)
    subprocess.run(['7z', 'x', '-y', z, f'-o{os.path.dirname(z)}'],
                   stdout=subprocess.DEVNULL)
!echo '--- 해제 후 구조 ---' && find {RAW_DIR} -maxdepth 4 -type d | head -40
!echo '--- json/wav 개수 ---' && echo "json: $(find {RAW_DIR} -name '*.json' | wc -l)" && echo "wav: $(find {RAW_DIR} -iname '*.wav' | wc -l)"

In [ ]:
# 5) 전처리 스크립트 가져오기(저장소 clone) 후 실행
![ -d translate ] && (cd translate && git pull -q) || git clone -q https://github.com/ysb2152/translate
# raw 구조가 세션 오디오+start/end 면 기본(슬라이스), 이미 발화 단위면 --no-slice 추가
!cd translate && python data/preprocess.py --raw {RAW_DIR} --out {OUT_ROOT} --val-ratio 0.05

In [ ]:
# 6) 산출물 확인
import json, itertools
print('=== stats ===')
print(open(f'{OUT_ROOT}/stats.json', encoding='utf-8').read())
print('\n=== STT 매니페스트 미리보기 ===')
for line in itertools.islice(open(f'{OUT_ROOT}/stt/train.jsonl', encoding='utf-8'), 3):
    print(line.strip())
print('\n=== 변환 문장쌍 미리보기 ===')
for line in itertools.islice(open(f'{OUT_ROOT}/mt/train.jsonl', encoding='utf-8'), 3):
    print(line.strip())

## 7. 디스크가 빠듯하면: 청크 단위 증분 처리

원본을 다 쌓지 말고, **한 청크 받기 → 전처리 → 원본 삭제 → 다음 청크**를 반복하면 로컬 사용량을 낮게 유지할 수 있다(우리 전처리는 작은 클립+매니페스트만 남긴다). 아래를 filekey 리스트를 돌며 반복 실행:

```python
for fk in ['44059', '44060', '44061']:
    !rm -rf {RAW_DIR}/* 
    !cd {RAW_DIR} && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {fk} -aihubapikey "{AIHUB_APIKEY}"
    # (압축 해제 셀) 실행
    !cd translate && python data/preprocess.py --raw {RAW_DIR} --out {OUT_ROOT} --val-ratio 0.05
    # 매니페스트는 append 가 아니라 덮어쓰므로, 청크별 out 을 나눠 만들고 나중에 합치는 방식 권장
```

> 참고: 현재 `preprocess.py`는 실행마다 매니페스트를 덮어쓴다. 증분으로 모으려면 청크별로 `--out data/processed_partN` 을 나눠 만든 뒤 jsonl 을 이어붙이면 된다(클립 경로는 절대경로라 병합해도 안전).

## 다음 단계 (B-8) — Whisper 파인튜닝

데이터(`{OUT_ROOT}/stt/*.jsonl`)가 준비되면:
1. **baseline 측정**: 표준 `whisper-base/small`로 val 셋 CER/WER 측정(사투리에서 얼마나 틀리는지).
2. **파인튜닝**: HuggingFace `transformers`로 Whisper를 방언 데이터에 파인튜닝.
3. **개선 측정**: 같은 val 셋에서 CER/WER 재측정 → 개선폭이 핵심 포트폴리오 지표.
4. **서빙 연결**: CTranslate2로 변환 후 백엔드 `WHISPER_MODEL_DIR`에 연결.

이 부분은 데이터가 실제로 준비된 뒤 별도 셀로 이어간다.